In [3]:
import json
import asyncio
import sys
from pathlib import Path

# Walk up from CWD to find the project root (identified by pyproject.toml),
# so imports work regardless of where Jupyter is launched from.
def find_project_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / "pyproject.toml").exists():
            return parent
    raise RuntimeError(f"Could not find project root from {start}")

project_root = find_project_root(Path().resolve())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from movie_ingestion.metadata_generation.inputs import MovieInputData
from movie_ingestion.metadata_generation.generators.plot_events import generate_plot_events

# Load all movies from the saved JSON
with open(project_root / "saved_imdb_movies.json") as f:
    raw_movies = json.load(f)

movie_tmdb_ids = [r["tmdb_id"] for r in raw_movies]

print(movie_tmdb_ids)

# Convert raw dicts to MovieInputData objects.
# The JSON uses debug_synopses/debug_plot_summaries (field names used during
# scraping) and release_date ("YYYY-MM-DD") instead of release_year.
def to_movie_input(raw: dict) -> MovieInputData:
    release_date = raw.get("release_date") or ""
    release_year = int(release_date[:4]) if release_date else None
    return MovieInputData(
        tmdb_id=raw["tmdb_id"],
        title=raw["title"],
        release_year=release_year,
        overview=raw.get("overview", ""),
        genres=raw.get("genres", []),
        plot_synopses=raw.get("debug_synopses", []),
        plot_summaries=raw.get("debug_plot_summaries", []),
        plot_keywords=raw.get("plot_keywords", []),
        overall_keywords=raw.get("overall_keywords", []),
        featured_reviews=raw.get("featured_reviews", []),
        reception_summary=raw.get("reception_summary"),
        audience_reception_attributes=raw.get("audience_reception_attributes", []),
        maturity_rating=raw.get("maturity_rating", ""),
        maturity_reasoning=raw.get("maturity_reasoning", []),
        parental_guide_items=raw.get("parental_guide_items", []),
    )

movies = [to_movie_input(r) for r in raw_movies]
print(f"Loaded {len(movies)} movies")


[9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
Loaded 50 movies


In [ ]:
ORIGINAL_SET_TMDB_IDS = [9377, 269149, 1584, 109445, 2493, 354912, 508965, 14160, 10674, 808, 13397, 76341, 85, 155, 245891, 1771, 569094, 299534, 11, 671, 120, 98, 27205, 603, 157336, 335984, 329, 329865, 493922, 694, 49018, 1034541, 176, 807, 496243, 419430, 1359, 550, 597, 13, 666277, 423, 11036, 1824, 25195, 216015, 392044, 545611, 22538, 37136]
MEDIUM_SPARSITY_TMDB_IDS = [329974, 1498832, 821937, 92, 160, 45739, 576560, 1383243, 1642210, 1639488]
HIGH_SPARSITY_TMDB_IDS = [270909, 493103, 64262, 1611977, 706910, 1297426, 35952, 158227, 215782, 1642486]

50


In [ ]:
# Run generate_plot_events on the first movie
first_movie = movies[0]
print(f"Running generate_plot_events for: {first_movie.title_with_year()}")

result, token_usage = await generate_plot_events(first_movie)
print(f"\nResult:\n{result}")
print(f"\nToken usage: {token_usage}")

In [ ]:
input_cost_pm = 0.125
output_cost_pm = 1
input_tokens_per_movie = 2737
output_tokens_per_movie = 953
total_movie_count = 112_000


input_cost = input_tokens_per_movie * total_movie_count * (input_cost_pm / 1_000_000)
output_cost = output_tokens_per_movie * total_movie_count * (output_cost_pm / 1_000_000)

total_cost = input_cost + output_cost

print(f"Total cost: ${total_cost:.2f}")


Total cost: $145.05
